In [ ]:
# Copyright (c) TorchGeo Contributors. All rights reserved.
# Licensed under the MIT License.

# Spatiotemporal Segmentation with ConvLSTM
_Written by: Yi-Chia Chang_

In this tutorial, we train a [Convolutional LSTM](https://arxiv.org/abs/1506.04214) from scratch for time-series semantic segmentation. ConvLSTM replaces the dense matrix multiplications inside a regular LSTM cell with 2D convolutions, so each gate operates on a `[C, H, W]` feature map instead of a flat vector — this preserves spatial structure while still modelling temporal dynamics.

Real-world applications:
- **Crop-type mapping**: different crops have distinctive phenological signatures that unfold across the growing season.
- **Forecasting**: short-horizon prediction of cloud cover, precipitation, sea ice, etc.

We train on [PASTIS100](https://torchgeo.readthedocs.io/en/stable/api/datasets/pastis.html) with Sentinel-2 time series over French agricultural parcels and a 20-class semantic mask. Unlike pretrained foundation models, ConvLSTM has no off-the-shelf weights for satellite imagery, so we train it end-to-end from random initialization.

## Setup

In [ ]:
%pip install torchgeo tensorboard

## Imports

In [ ]:
%matplotlib inline
%load_ext tensorboard

import os
import tempfile

import lightning as L
import matplotlib.pyplot as plt
import torch
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

from torchgeo.datamodules import PASTIS100DataModule
from torchgeo.datasets import PASTIS100
from torchgeo.models import ConvLSTM
from torchgeo.trainers import SemanticSegmentationTask

torch.set_float32_matmul_precision('medium')
L.seed_everything(0, workers=True)

## Visualize the dataset

PASTIS100 ships 100 patches at 128×128 with a variable-length Sentinel-2 time series per patch and a static 20-class semantic mask (background + 18 crop classes + a *void* label for parcels mostly outside the patch). We use all 10 L2A bands.

In [ ]:
root = os.path.join(tempfile.gettempdir(), 'pastis100')
dataset = PASTIS100(root=root, bands=PASTIS100.s2_bands, download=True)

sample = dataset[2]
fig = dataset.plot(sample, suptitle='PASTIS100 sample')
plt.show()

print(f'Dataset size: {len(dataset)} patches')
print(f'Image shape: {sample["image"].shape} (T, C, H, W)')
print(f'Mask shape: {sample["mask"].shape} (H, W)')

## DataModule

`PASTIS100DataModule` downloads the data, builds the train/val/test split, and pads each time series to a common length. We override its augmentation to a single divide-by-10000 normalization, which puts Sentinel-2 reflectance roughly in `[0, 1]`.

In [ ]:
batch_size = 4
num_workers = 4
max_epochs = 30
padding_length = 30
fast_dev_run = False

In [ ]:
import kornia.augmentation as K

datamodule = PASTIS100DataModule(
    root=root,
    bands=PASTIS100.s2_bands,
    batch_size=batch_size,
    num_workers=num_workers,
    padding_length=padding_length,
    download=True,
)
datamodule.aug = K.AugmentationSequential(
    K.VideoSequential(K.Normalize(mean=torch.tensor(0.0), std=torch.tensor(10000.0))),
    data_keys=None,
    keepdim=True,
)

## Training

Trainer WIP

In [ ]:
trainer.fit(model=task, datamodule=datamodule)

## Inference

We use the model trained with PASTIS data and inference on new regions. In this case, we will use Germnany as an example as it has crop type labels in EuroCrops and Germany is not included in our training dataset, PASTIS. 

## Evaluation

Evalutate the inference results using EuroCrops' labels